# 06 — Huấn luyện, chấm điểm và so sánh

Tương ứng src/train.py. Đọc dữ liệu, kiểm tra Train/Test, xem cấu hình từng mô hình, chạy huấn luyện rồi đọc hai bảng điểm riêng.

**Cách chạy:** chọn kernel Python rồi chạy từng ô từ trên xuống (Shift+Enter), hoặc chọn **Run All**.

**Cần có trước:** Snapshot giá và bảng bổ trợ đã chuẩn hóa (notebook 02). Không cần GPU.

Các ô gọi hàm thật trong `src/`, không gọi `main()` nhận tham số của kernel và không chép lại thuật toán. Mỗi lần chạy có thể cập nhật tệp kết quả trong `data/processed/`, `reports/`, `figures/`, `models/`. Dữ liệu nghiên cứu dừng tại 30/09/2025.

Xem [hướng dẫn](README.md) nếu chạy trên Google Colab hoặc gặp lỗi kernel.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## 4. Nạp cùng cohort cho cả hai nhánh

Cohort nghĩa là cùng nhóm ngày có đủ dữ liệu. Cả hai nhánh cùng nhãn, cùng mốc chia và cùng giai đoạn Test.

In [ ]:
from prepare import build_experiment_table
base, common, branches = build_experiment_table(PRICE_PATH, SUPPLEMENTAL_PATH)
display(pd.DataFrame([{"Nhánh": name, "Số biến": len(cols), "Dòng đầu vào": len(common)} for name, cols in branches.items()]))

## 5. Xem Train/Test trước khi fit

Không xáo trộn. Train gồm 85% hàng đầu theo ngày nhãn. Phiên đứng giá bị loại riêng khỏi bài toán phân loại.

In [ ]:
from train import _chronological_splits
masks, boundary = _chronological_splits(common)
display(pd.Series(boundary).to_frame("Train/Test"))
partition_rows = []
for part, mask in masks.items():
    subset = common.loc[mask]
    partition_rows.append({
        "Tập": part, "Mẫu hồi quy": len(subset),
        "Mẫu phân loại": int(subset["direction"].notna().sum()),
        "Ngày nhãn đầu": subset["target_date"].min(),
        "Ngày nhãn cuối": subset["target_date"].max(),
    })
display(pd.DataFrame(partition_rows))
assert common.loc[masks["train"], "target_date"].max() < common.loc[masks["test"], "target_date"].min()

## 6. Xem mô hình và siêu tham số

Majority đoán lớp phổ biến; Logistic học trọng số; Random Forest gộp nhiều cây; XGBoost/HGB thêm cây để sửa lỗi. Hồi quy dùng các họ tương tự, thêm Last-Close lấy giá hôm nay làm dự báo ngày mai. Các tham số đặt trước, không chỉnh theo Test.

**Lưu ý khi chỉnh tham số:** các đối tượng ở ô dưới chỉ để xem cấu hình. `run_training` tạo lại mô hình từ `src/train.py`; nếu muốn đổi siêu tham số thật, sửa `_classifier_models()` / `_regressor_models()` trong file đó rồi khởi động lại kernel và chạy lại từ đầu.

In [ ]:
from train import _classifier_models, _regressor_models
classifier_candidates = _classifier_models()
regressor_candidates = _regressor_models()
display(pd.DataFrame([{"Bài toán": "Phân loại", "Mô hình": name, "Cấu hình": str(model)} for name, model in classifier_candidates.items()]))
display(pd.DataFrame([{"Bài toán": "Hồi quy", "Mô hình": name, "Cấu hình": str(model)} for name, model in regressor_candidates.items()]))
print("Hồi quy còn có Last-Close Baseline: predicted_close = current_close, không cần fit.")
display(classifier_candidates["Logistic Regression"].get_params())

## 7. Chạy toàn bộ huấn luyện và xuất artifact

Hàm gốc fit mô hình trên Train, chấm Test, lưu đủ bảng và hình, chọn mô hình minh họa rồi refit toàn bộ nhãn. Việc chọn theo Test làm đánh giá sau chọn mang tính thăm dò. Không dùng artifact refit để chấm lại bảng Test.

In [ ]:
from train import run_training
# Cùng hàm dùng ở CLI: fit Train, chấm Test, xuất bảng/ảnh, rồi refit mô hình được chọn.
# CPU đủ cho dữ liệu này. Chờ ô hoàn tất, không bấm chạy trùng lần hai.
training_summary = run_with_log(
    run_training, PRICE_PATH, FIGURES_DIR, REPORTS_DIR, MODELS_DIR, PROCESSED_DIR, SUPPLEMENTAL_PATH
)
display(pd.DataFrame({
    "Phân loại": training_summary["classifier_selected"],
    "Hồi quy": training_summary["regressor_selected"],
}))

## 8. Bảng điểm phân loại và confusion matrix

Accuracy/BA/Precision/Recall/F1/AUC là tỷ lệ [0,1]. Confusion matrix cho số phiên đoán đúng/sai của hai lớp; 0=Giảm và 1=Tăng.

In [ ]:
classification = pd.read_csv(REPORTS_DIR / "classification_test_metrics.csv")
display(classification[["feature_set", "model", "test_rows", "accuracy", "balanced_accuracy", "precision_up", "recall_up", "f1_up", "roc_auc"]].round(4))
# Các tỷ lệ trên nằm trong [0, 1]; nhân 100 nếu muốn đọc theo phần trăm.
display(pd.read_csv(REPORTS_DIR / "confusion_matrices.csv"))

## 9. Bảng điểm giá và dung sai

MAE/RMSE càng nhỏ càng tốt, đơn vị nghìn VND. Tolerance ±0,5%/±1%/±2% càng lớn càng tốt và không đồng nghĩa đoán đúng chiều. MAPE/DA/tolerance đã theo phần trăm.

In [ ]:
regression = pd.read_csv(REPORTS_DIR / "regression_test_metrics.csv")
display(regression[["feature_set", "model", "mae", "rmse", "mape_percent", "directional_accuracy_nonflat_percent",
                    "tolerance_accuracy_0_5_percent", "tolerance_accuracy_1_percent", "tolerance_accuracy_2_percent"]].round(4))
display(pd.read_csv(REPORTS_DIR / "ablation_summary.csv"))

## 10. Đường thực tế và dự báo

Xem tất cả mô hình và hai nhánh trên cùng Test; biểu đồ thứ hai phóng to 80 phiên cuối. Cây khó ngoại suy ngoài vùng giá Train, vì vậy cần giữ baseline để đối chiếu.

In [ ]:
for name in ["07_regression_lines_by_branch.png", "08_branch_xgboost_comparison_zoom.png", "11_all_regressors_branch_comparison_zoom.png"]:
    show_figure(name)

## 11. Biểu đồ cột và ma trận nhầm lẫn

Không chỉ nhìn một Accuracy cao nhất: kiểm tra BA/F1 và sai số giá, đồng thời so cùng thuật toán giữa hai nhánh.

In [ ]:
for name in ["09_classification_metric_bars.png", "10_regression_metric_bars.png", "05_all_confusion_matrices.png"]:
    show_figure(name)

## 12. Kiểm tra metadata của mô hình đã lưu

joblib lưu mô hình và tên cột đầu vào. Chỉ nạp artifact của dự án mà bạn tin cậy. Mô hình đã refit dùng cho dự báo mới nhất, khác bản Train-only dùng chấm Test.

In [ ]:
import joblib
artifact_rows = []
for filename in ["best_classifier.joblib", "best_regressor.joblib"]:
    artifact = joblib.load(MODELS_DIR / filename)
    artifact_rows.append({"Tệp": filename, **{key: value for key, value in artifact.items() if key not in {"model", "feature_columns"}}, "Số đầu vào": len(artifact["feature_columns"])})
display(pd.DataFrame(artifact_rows))